In [ ]:
# ===============================
# XLM-R HATE SPEECH DETECTION (PRO COLAB PIPELINE)
# + FOCAL LOSS + CLASS IMBALANCE HANDLING
# ===============================

# ---------- 1. INSTALL ----------
!pip install transformers torch scikit-learn pandas numpy openpyxl

# ---------- 2. IMPORTS ----------
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 3. LOAD DATASET (CSV / XLSX AUTO) ----------

def load_data(path):
    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        raise ValueError("Only CSV or XLSX supported")

    df = df.dropna().drop_duplicates()

    # auto-detect columns
    text_col, label_col = df.columns[0], df.columns[1]

    return df[text_col].astype(str), df[label_col].astype(str)

TEXTS, LABELS = load_data("/content/All_annotated_final.xlsx")  # change file

# ---------- 4. LABEL ENCODING ----------
le = LabelEncoder()
y = le.fit_transform(LABELS)

X_train, X_test, y_train, y_test = train_test_split(
    TEXTS, y, test_size=0.2, random_state=42, stratify=y
)

# ---------- 5. CLASS IMBALANCE ----------
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# ---------- 6. TOKENIZER ----------
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ---------- 7. DATASET ----------
class HateDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = HateDataset(X_train, y_train)
test_dataset = HateDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# ---------- 8. MODEL ----------
model = AutoModel.from_pretrained(model_name)
model.to(DEVICE)

classifier = nn.Linear(768, len(np.unique(y))).to(DEVICE)

# ---------- 9. FOCAL LOSS ----------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=alpha)

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        focal = (1 - pt) ** self.gamma * ce_loss
        return focal

criterion = FocalLoss(alpha=class_weights)
optimizer = torch.optim.Adam(list(model.parameters()) + list(classifier.parameters()), lr=2e-5)

# ---------- 10. TRAIN LOOP ----------
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    classifier.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]

        logits = classifier(cls)

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

# ---------- 11. EVALUATION ----------
model.eval()
classifier.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]

        logits = classifier(cls)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- 12. METRICS ----------
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1 Score (weighted):", f1_score(all_labels, all_preds, average="weighted"))
print("Precision (weighted):", precision_score(all_labels, all_preds, average="weighted"))
print("Recall (weighted):", recall_score(all_labels, all_preds, average="weighted"))
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

# ---------- DONE ----------
print("Training complete - XLM-R + Focal Loss pipeline")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 Loss: 0.1411
Epoch 2 Loss: 0.1075
Epoch 3 Loss: 0.1165
Accuracy: 0.793233082706767
F1 Score (weighted): 0.8117954221759459
Precision (weighted): 0.8437302451853445
Recall (weighted): 0.793233082706767

Classification Report:
              precision    recall  f1-score   support

          NO       0.92      0.82      0.87      1584
           O       0.38      0.62      0.47       278

    accuracy                           0.79      1862
   macro avg       0.65      0.72      0.67      1862
weighted avg       0.84      0.79      0.81      1862

Training complete - XLM-R + Focal Loss pipeline


### Comparing different Transformer models

Now, let's run the hate speech detection pipeline with various pre-trained Transformer models and compare their performance metrics. We'll use the following models:

*   `xlm-roberta-base` (already run)
*   `google/muril-base-cased`
*   `ai4bharat/indic-bert`
*   `bert-base-multilingual-cased`
*   `bert-base-uncased`

We'll capture Accuracy, F1 Score (weighted), Precision (weighted), and Recall (weighted) for each model.

In [ ]:
# List of models to compare
model_comparison_names = [
    "xlm-roberta-base", # Already run, but including for completeness
    "google/muril-base-cased",
    "ai4bharat/indic-bert",
    "bert-base-multilingual-cased",
    "l3cube-pune/hindi-bert-v2"
]

# List to store results for each model
results = []


In [ ]:
def train_and_evaluate_model(model_name, X_train, y_train, X_test, y_test, le, class_weights, DEVICE, EPOCHS=3, batch_size=16):
    print(f"\n--- Training and evaluating {model_name} ---")

    # Re-initialize tokenizer and model for each run
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to(DEVICE)

    # Re-initialize classifier (adjust input features if needed, assuming 768 for most base models)
    classifier = nn.Linear(model.config.hidden_size, len(np.unique(y_train))).to(DEVICE)

    # Re-define HateDataset within the function to ensure the correct tokenizer is used
    class HateDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_length=128):
            self.texts = list(texts)
            self.labels = list(labels)
            self.tokenizer = tokenizer
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            text = self.texts[idx]
            label = self.labels[idx]

            encoding = self.tokenizer(
                text,
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt"
            )

            return {
                "input_ids": encoding["input_ids"].squeeze(),
                "attention_mask": encoding["attention_mask"].squeeze(),
                "labels": torch.tensor(label, dtype=torch.long)
            }

    train_dataset = HateDataset(X_train, y_train, tokenizer)
    test_dataset = HateDataset(X_test, y_test, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    # Focal Loss and Optimizer
    # criterion (FocalLoss) is defined in the previous cell and accessible globally
    criterion = FocalLoss(alpha=class_weights)
    optimizer = torch.optim.Adam(list(model.parameters()) + list(classifier.parameters()), lr=2e-5)

    # Train loop
    for epoch in range(EPOCHS):
        model.train()
        classifier.train()
        total_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            cls = outputs.last_hidden_state[:, 0, :]
            logits = classifier(cls)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation
    model.eval()
    classifier.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            cls = outputs.last_hidden_state[:, 0, :]
            logits = classifier(cls)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")
    precision_weighted = precision_score(all_labels, all_preds, average="weighted")
    recall_weighted = recall_score(all_labels, all_preds, average="weighted")

    # Store classification report as a string for later inspection if needed
    class_report_str = classification_report(all_labels, all_preds, target_names=le.classes_, output_dict=False)

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "F1 Score (weighted)": f1_weighted,
        "Precision (weighted)": precision_weighted,
        "Recall (weighted)": recall_weighted,
        "Classification Report": class_report_str
    }
